In [1]:
import numpy as np

In [2]:
data = "hello"

chars = list(set(data)) #unique char
vocab_size = len(chars) 

In [3]:
# mappings
char_to_ix = {ch:i for i,ch in enumerate(chars)}  # eg{'h':0,'e':1,'l':2,'o':3}
ix_to_char = {i:ch for i,ch in enumerate(chars)}

In [4]:
print("data:", data)
print("chars:", chars)
print("vocab_size:", vocab_size)

data: hello
chars: ['o', 'e', 'l', 'h']
vocab_size: 4


In [5]:
#vector rep
def one_hot(ix):
    x = np.zeros((vocab_size, 1))
    x[ix] = 1
    return x

print(one_hot(char_to_ix['e']))

[[0.]
 [1.]
 [0.]
 [0.]]


In [6]:
# initialize weights

hidden_size = 10  #neurons 

Wxh = np.random.randn(hidden_size, vocab_size) * 0.01  #input->hidden weights
Whh = np.random.randn(hidden_size, hidden_size) * 0.01 #hidden-to-hidden (recurrent) weights
Why = np.random.randn(vocab_size, hidden_size) * 0.01  #hidden-to-output weights

bh = np.zeros((hidden_size, 1)) #biases
by = np.zeros((vocab_size, 1)) 


In [7]:
# forward pass
def forward(inputs, hprev):
    h = hprev  #hidden layer from the previous time step
    
    for ch in inputs:
        x = one_hot(char_to_ix[ch])  # char into one hot vector
        h = np.tanh(Wxh @ x + Whh @ h + bh)  # @ -> np.dot( matrix mult)
    
    y = Why @ h + by  # ourput
    p = np.exp(y) / np.sum(np.exp(y))  # for softmax
    
    return p, h  # probabilities for next character , final hidden layer

In [8]:
hprev = np.zeros((hidden_size, 1))
p, h = forward("hello", hprev)

print("probabilities:", p)

probabilities: [[0.24996057]
 [0.24994737]
 [0.25002487]
 [0.2500672 ]]


In [9]:
# pred
def predict(seed):
    h = np.zeros((hidden_size, 1))
    
    for ch in seed:
        x = one_hot(char_to_ix[ch])
        h = np.tanh(Wxh @ x + Whh @ h + bh)
    
    y = Why @ h + by
    p = np.exp(y) / np.sum(np.exp(y))
    
    return ix_to_char[np.argmax(p)]

In [10]:
print(predict("hell"))

h


In [11]:
    def lossFun(inputs, targets, hprev):
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = np.copy(hprev)
        loss = 0
    
        # Forward pass
        for t in range(len(inputs)):
            xs[t] = one_hot(char_to_ix[inputs[t]])
            hs[t] = np.tanh(Wxh @ xs[t] + Whh @ hs[t-1] + bh)
            ys[t] = Why @ hs[t] + by
            ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
            loss += -np.log(ps[t][char_to_ix[targets[t]], 0])
    
    
        # Backward pass
        dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
        dbh, dby = np.zeros_like(bh), np.zeros_like(by)
        dhnext = np.zeros_like(hs[0])
    
        for t in reversed(range(len(inputs))):
            dy = np.copy(ps[t])
            dy[char_to_ix[targets[t]]] -= 1
    
            dWhy += dy @ hs[t].T
            dby += dy
    
            dh = Why.T @ dy + dhnext
            dhraw = (1 - hs[t]**2) * dh
    
            dbh += dhraw
            dWxh += dhraw @ xs[t].T
            dWhh += dhraw @ hs[t-1].T
    
            dhnext = Whh.T @ dhraw
    
    
             #grad
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)
    
        return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

In [12]:
    learning_rate = 0.1
    
    hprev = np.zeros((hidden_size, 1))
    
    inputs = "hell"
    targets = "ello"


'''
inputs->targets
h → e
e → l
l → l
l → o
'''
    
    for i in range(500):
        loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
    
        # update weights
        for param, dparam in zip([Wxh, Whh, Why, bh, by],
                                 [dWxh, dWhh, dWhy, dbh, dby]):
            param -= learning_rate * dparam
    
        if i % 100 == 0:
            print("iteration", i, "loss:", loss)

iteration 0 loss: 5.544634816192293
iteration 100 loss: 0.06454212541187354
iteration 200 loss: 0.021199899739408348
iteration 300 loss: 0.012499033107343831
iteration 400 loss: 0.00881230995816126


In [13]:
print("Prediction:", predict("hell"))

Prediction: o
